In [23]:
!pip install yfinance hmmlearn tensorflow scikit-learn


In [24]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from datetime import datetime

from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


In [25]:
symbol = "^NSEI"

start = "2010-01-01"
end = datetime.today().strftime("%Y-%m-%d")

print("Downloading till:", end)

df = yf.download(symbol, start=start, end=end)


/tmp/ipython-input-2134946324.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=start, end=end)
[*********************100%***********************]  1 of 1 completed

In [26]:
df["Return"] = df["Close"].pct_change()

df["Volatility"] = df["Return"].rolling(20).std()

# Safe Log Volume
df["LogVolume"] = np.log(df["Volume"].replace(0, 1))

df["MA20"] = df["Close"].rolling(20).mean()
df["MA50"] = df["Close"].rolling(50).mean()

df["MA_Diff"] = df["MA20"] - df["MA50"]


# RSI (Safe)
delta = df["Close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / (avg_loss + 1e-10)

df["RSI"] = 100 - (100 / (1 + rs))


# Remove bad values
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

print("Rows after cleaning:", len(df))


Rows after cleaning: 3909


In [27]:
features = [
    "Return",
    "Volatility",
    "LogVolume",
    "MA_Diff",
    "RSI"
]

X = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


print("Any NaN:", np.isnan(X_scaled).sum())
print("Any inf:", np.isinf(X_scaled).sum())


Any NaN: 0
Any inf: 0


In [28]:
def train_hmm(n_states, seed):

    model = GaussianHMM(
        n_components=n_states,
        covariance_type="diag",   # FIX
        min_covar=1e-4,           # FIX
        n_iter=500,
        random_state=seed
    )

    model.fit(X_scaled)

    return model


hmm1 = train_hmm(3, 1)
hmm2 = train_hmm(4, 2)
hmm3 = train_hmm(5, 3)



In [29]:
s1 = hmm1.predict(X_scaled)
s2 = hmm2.predict(X_scaled)
s3 = hmm3.predict(X_scaled)


def majority_vote(a, b, c):

    return np.array([
        max(set(x), key=x.count)
        for x in zip(a,b,c)
    ])


df["Regime"] = majority_vote(s1, s2, s3)


In [ ]:
def rolling_hmm(window=2000):

    regimes = []

    for i in range(window, len(X_scaled)):

        subX = X_scaled[i-window:i]

        model = GaussianHMM(
            n_components=3,        # FIX
            covariance_type="diag",# FIX
            min_covar=1e-4,        # FIX
            n_iter=200,
            random_state=42
        )

        model.fit(subX)

        state = model.predict(
            X_scaled[i].reshape(1,-1)
        )[0]

        regimes.append(state)

    return np.array(regimes)


rolling_regimes = rolling_hmm()


In [ ]:
df["Tomorrow"] = (df["Return"].shift(-1) > 0).astype(int)

df = df.dropna()


In [ ]:
def build_lstm():

    model = Sequential()

    model.add(LSTM(64, return_sequences=True,
                   input_shape=(30, len(features))))

    model.add(Dropout(0.3))

    model.add(LSTM(64))

    model.add(Dropout(0.3))

    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


In [ ]:
window = 30

lstm_models = {}

for r in df["Rolling_Regime"].unique():

    sub = df[df["Rolling_Regime"] == r]

    X_lstm = []
    y_lstm = []

    data_reg = sub[features].values

    scaler_lstm = MinMaxScaler()
    data_reg = scaler_lstm.fit_transform(data_reg)

    for i in range(window, len(data_reg)):

        X_lstm.append(data_reg[i-window:i])
        y_lstm.append(sub["Tomorrow"].iloc[i])


    X_lstm = np.array(X_lstm)
    y_lstm = np.array(y_lstm)

    if len(X_lstm) < 200:
        continue


    model = build_lstm()

    model.fit(
        X_lstm,
        y_lstm,
        epochs=15,
        batch_size=16,
        verbose=0
    )

    lstm_models[r] = (model, scaler_lstm)


KeyError: 'Rolling_Regime'

In [ ]:
def predict_next():

    r = df["Rolling_Regime"].iloc[-1]

    if r not in lstm_models:
        return None


    model, scaler = lstm_models[r]


    last = df[features].iloc[-30:].values

    last = scaler.transform(last)

    last = last.reshape(1,30,len(features))


    prob = model.predict(last)[0][0]

    return prob


In [ ]:
p = predict_next()

print("Last Date:", df.index[-1].date())
print("Probability Market Goes Up:", round(p*100,2), "%")


if p > 0.6:
    print("Signal: STRONG BUY")

elif p > 0.55:
    print("Signal: BUY")

elif p < 0.4:
    print("Signal: SELL")

else:
    print("Signal: HOLD")
